In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1" #1 for rk8

In [2]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import (
    process_images,
    tokenizer_image_token,
    get_model_name_from_path,
)
from llava.eval.run_llava import load_images
from llava.constants import IMAGE_TOKEN_INDEX

from tqdm import tqdm

import glob
import torch
import numpy as np

import json
import argparse

[2024-07-12 19:17:56,576] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [3]:
torch.cuda.device_count()

1

In [4]:
prompt = "This is the poster of a movie. What is the genre of the movie? Describe the picture. What feelings does it convey?"
model_path = "liuhaotian/llava-v1.6-34b"
images_path = "/share/hel/datasets/mmimdb/dataset/*.jpeg"
encoded_data_path = "/share/hel/datasets/mmimdb/llava_encoded_images/"
json_path = os.path.join(encoded_data_path, f'decoded_texts.json')

In [5]:
def load_model(
        # device_map="auto", device="cuda", use_flash_attn=False, 
        model_path,
        model_base=None,
        model_name=get_model_name_from_path(model_path),
        # device='cpu'
        load_4bit=True,
):

    tokenizer, model, image_processor, context_len = load_pretrained_model( # device_map="auto", device="cuda", use_flash_attn=False, **kwargs
        model_path=model_path,
        model_base=model_base,
        model_name=get_model_name_from_path(model_path),
        # device='cpu'
        load_4bit=load_4bit,
    )
    
    return tokenizer, model, image_processor, context_len

In [6]:
def get_image_files(
        images_path,
        limit=None):
    image_files = glob.glob(images_path)[:limit] # This will work also if limit=None, and take the full list.
    image_names = [os.path.basename(x).split('.')[0] for x in image_files]
    
    return image_names, image_files

In [7]:
def generate_prompt_ids(
        prompt,
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
):
    input_ids = (
        tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors)
        .unsqueeze(0)
        .cuda()
    )
    return input_ids

In [8]:
def process_image(
        image_file,
        model,
        image_processor,
):
    image = load_images([image_file])
    image_size = image[0].size
    image_tensor = process_images(
        image,
        image_processor,
        model.config
    ).to(model.device, dtype=torch.float16)
    return image_size, image_tensor

In [9]:
def generate_from_image_and_prompt(
        image_size,
        image_tensor,
        model,
        tokenizer,
        input_ids,
        max_new_tokens=512,
        use_cache=False,
):
    # Generate from both prompt and image
    with torch.inference_mode():
        output_ids = model.generate(
            inputs=input_ids,
            images=image_tensor,
            image_sizes=image_size,
            # do_sample=False, # True if args.temperature > 0 else False,
            # temperature=0, # args.temperature,
            # top_p=args.top_p,
            # num_beams=args.num_beams,
            max_new_tokens=max_new_tokens,
            use_cache=use_cache,
            tokenizer=tokenizer,
        )
        # print(output_ids)
        outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    return outputs

In [10]:
def get_last_layer_from_image_and_prompt(
        image_size,
        image_tensor,
        model,
        tokenizer,
        input_ids,
):
    # Generate from both prompt and image
    with torch.inference_mode():
        outputs = model.get_last_layer(
            inputs=input_ids,
            images=image_tensor,
            image_sizes=image_size,
            #TODO maybe pass output_hidden_states as in line 1077
            # https://github.com/huggingface/transformers/blob/d1a1bcf56aeb8593b9cc613b21422e6311875599/src/transformers/models/llama/modeling_llama.py#L1077C13-L1077C54
            # and add it to the super().forward argument
        )
        # print(output_ids)
        # outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    return outputs

In [11]:
tokenizer, model, image_processor, context_len = load_model(model_path)

/home/marta/miniconda3/envs/flashllava/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/home/marta/miniconda3/envs/flashllava/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

In [13]:
model

LlavaLlamaForCausalLM(
  (model): LlavaLlamaModel(
    (embed_tokens): Embedding(64000, 7168, padding_idx=0)
    (layers): ModuleList(
      (0-59): 60 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=7168, out_features=7168, bias=False)
          (k_proj): Linear4bit(in_features=7168, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=7168, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=7168, out_features=7168, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=7168, out_features=20480, bias=False)
          (up_proj): Linear4bit(in_features=7168, out_features=20480, bias=False)
          (down_proj): Linear4bit(in_features=20480, out_features=7168, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )


In [24]:
import inspect

In [57]:
output_tensors[1].shape

(1, 7168)

In [12]:
image_names, image_files = get_image_files(images_path, limit=None)
input_ids = generate_prompt_ids(prompt, tokenizer, IMAGE_TOKEN_INDEX)
output_tensors = []

for image_name, image_file in tqdm(zip(image_names, image_files)):
    image_size, image_tensor = process_image(image_file, model, image_processor)
    outputs = get_last_layer_from_image_and_prompt(
        image_size,
        image_tensor,
        model,
        tokenizer,
        input_ids
    )
    output_tensor = outputs.hidden_states[-1].cpu().detach().numpy().mean(axis=1)
    output_tensors.append(output_tensor)


feature_file_path = os.path.join(encoded_data_path, 'llava_latent_tensors.npz')
np.savez(feature_file_path, indices=image_names, values=output_tensors)
print('done!')

1it [00:05,  5.35s/it]


KeyboardInterrupt: 

In [19]:
outputs_texts_dict

{'0043511': 'The image is a poster for the movie "Europe \'51". The movie is directed by Roberto Rossellini. The poster features a woman in a black coat and a white scarf, standing in front of a city skyline. The movie is part of the Criterion Collection and was released in 1952. The poster conveys a sense of drama and intrigue, suggesting that the movie is likely to be a serious and thought-provoking piece of cinema.',
 '0489270': 'The image is a movie poster for the film "Saul". The poster features a man in a red hooded cloak, standing in a dark room. The man is looking directly at the viewer, creating a sense of connection. The background is black, which contrasts with the red of the cloak and the white text at the top of the poster. The text reads "Legends never die", suggesting that the film may be about a legendary figure or a story of enduring legacy. The overall mood of the poster is mysterious and intense, hinting at the drama and intrigue that the film may contain.',
 '154457

In [32]:
with open(json_path, 'w') as fp:
    json.dump(outputs_last_layers_dict, fp)

TypeError: Object of type Tensor is not JSON serializable

In [ ]:
image_names, image_files = get_image_files(images_batch, limit=None)
input_ids = generate_prompt_ids(prompt, tokenizer, IMAGE_TOKEN_INDEX)
outputs_texts_dict = {}

for image_name, image_file in tqdm(zip(image_names, image_files)):
    image_size, image_tensor = process_image(image_file, model, image_processor)
    outputs = generate_from_image_and_prompt(
        image_size,
        image_tensor,
        model,
        tokenizer,
        input_ids
    )
    outputs_texts_dict[image_name] = outputs

In [24]:
with open(json_path, 'w') as fp:
    json.dump(outputs_texts_dict, fp)

In [ ]:
# TODO adapt this for the last layer of the LLM

    image_files = glob.glob(images_path)
    image_names = [os.path.basename(x).split('.')[0] for x in image_files]
    # print(f'Encoding {one_image_path}...')
    # Not sure if I should follow run_llava line 100 on
    # or model_vqa to encode the image
    image_tensors = []
    for image_file in tqdm(image_files):
        image = load_images([image_file])
        image_tensor = process_images(
            image,
            image_processor,
            model.config
        ).to(model.device, dtype=torch.float16)

        image_tensor = model.encode_images(image_tensor)
        image_tensor = image_tensor.cpu().detach().numpy()
        image_tensors.append(image_tensor)

    # images = load_images(image_files)
    # print(images)
    # feature_file_path = os.path.join(encoded_data_path, 'llava_images.npz')
    # np.savez(feature_file_path, indices=image_names, values=images.cpu())
    #
    # image_tensors = process_images(
    #     images,
    #     image_processor,
    #     model.config
    # ).to(model.device, dtype=torch.float16)
    #
    # image_tensors = model.encode_images(image_tensors)

    feature_file_path = os.path.join(encoded_data_path, 'llava_image_tokens.npz')
    np.savez(feature_file_path, indices=image_names, values=image_tensors)